In [1]:
import pandas as pd
import numpy as np
import requests
import ast

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [2]:
def clean_team(x):
    try:
        x = str(x)
        while isinstance(x, str):
            try:
                x = ast.literal_eval(x)
            except:
                break
        if isinstance(x, str):
            x = x.replace('[','').replace(']','')
            x = [int(i.strip()) for i in x.split(',') if i.strip().isdigit()]
        return list(x)
    except:
        return []

df = pd.read_csv("matches.csv")

df['radiant_team'] = df['radiant_team'].apply(clean_team)
df['dire_team'] = df['dire_team'].apply(clean_team)

df = df[
    (df['radiant_team'].apply(lambda x: len(x) == 5 and all(h != 0 for h in x))) &
    (df['dire_team'].apply(lambda x: len(x) == 5 and all(h != 0 for h in x)))
]

In [3]:
from collections import Counter

hero_counts = Counter()

for _, row in df.iterrows():
    hero_counts.update(row['radiant_team'])
    hero_counts.update(row['dire_team'])

MIN_GAMES = 500

valid_heroes = {
    h for h, count in hero_counts.items()
    if count >= MIN_GAMES
}

df = df[
    df['radiant_team'].apply(lambda team: all(h in valid_heroes for h in team)) &
    df['dire_team'].apply(lambda team: all(h in valid_heroes for h in team))
]

In [4]:
heroes = sorted(list(valid_heroes))
hero_to_idx = {h: i for i, h in enumerate(heroes)}
NUM_HEROES = len(heroes)

In [5]:
matchups = []

for _, row in df.iterrows():
    r = row['radiant_team']
    d = row['dire_team']
    win = row['radiant_win']

    for a in r:
        for b in d:
            matchups.append((a, b, win))
            matchups.append((b, a, 1 - win))

matchups_df = pd.DataFrame(matchups, columns=['hero_1', 'hero_2', 'win'])

In [6]:
winrates = matchups_df.groupby(['hero_1', 'hero_2'])['win'].mean().reset_index()

strong_counters = winrates[
    (winrates['win'] < 0.35) | (winrates['win'] > 0.65)
]

In [7]:
counter_pairs = list(zip(strong_counters['hero_1'], strong_counters['hero_2']))
pair_to_idx = {pair: i for i, pair in enumerate(counter_pairs)}

NUM_COUNTERS = len(counter_pairs)

In [8]:
X = []
y = []

for _, row in df.iterrows():
    vec = np.zeros(NUM_HEROES + NUM_COUNTERS)

    for h in row['radiant_team']:
        vec[hero_to_idx[h]] = 1
    for h in row['dire_team']:
        vec[hero_to_idx[h]] = -1

    offset = NUM_HEROES

    for r in row['radiant_team']:
        for d in row['dire_team']:
            if (r, d) in pair_to_idx:
                vec[offset + pair_to_idx[(r, d)]] += 2
            if (d, r) in pair_to_idx:
                vec[offset + pair_to_idx[(d, r)]] -= 2

    X.append(vec)
    y.append(row['radiant_win'])

X = np.array(X)
y = np.array(y)

In [9]:
X = []
y = []

for _, row in df.iterrows():
    vec = np.zeros(NUM_HEROES + NUM_COUNTERS)

    for h in row['radiant_team']:
        vec[hero_to_idx[h]] = 1
    for h in row['dire_team']:
        vec[hero_to_idx[h]] = -1

    offset = NUM_HEROES

    for r in row['radiant_team']:
        for d in row['dire_team']:
            if (r, d) in pair_to_idx:
                vec[offset + pair_to_idx[(r, d)]] += 2
            if (d, r) in pair_to_idx:
                vec[offset + pair_to_idx[(d, r)]] -= 2

    X.append(vec)
    y.append(row['radiant_win'])

X = np.array(X)
y = np.array(y)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.560196025047645


In [11]:
def predict(hero, enemy_team, my_team):
    vec = np.zeros(NUM_HEROES + NUM_COUNTERS)

    team = my_team + [hero]

    for h in team:
        vec[hero_to_idx[h]] = 1
    for h in enemy_team:
        vec[hero_to_idx[h]] = -1

    offset = NUM_HEROES

    for r in team:
        for d in enemy_team:
            if (r, d) in pair_to_idx:
                vec[offset + pair_to_idx[(r, d)]] += 2
            if (d, r) in pair_to_idx:
                vec[offset + pair_to_idx[(d, r)]] -= 2

    return model.predict_proba([vec])[0][1]

In [12]:
def suggest(enemy_team, my_team=[]):
    taken = set(enemy_team + my_team)
    candidates = [h for h in hero_to_idx.keys() if h not in taken]

    results = []

    for hero in candidates:
        prob = predict(hero, enemy_team, my_team)

        # 🔥 TEAM-AWARE MATCHUP LOGIC
        matchup_scores = []

        for e in enemy_team:
            row = strong_counters[
                (strong_counters['hero_1'] == hero) &
                (strong_counters['hero_2'] == e)
            ]

            if not row.empty:
                winrate = row['win'].values[0]
                matchup_scores.append(winrate - 0.5)

        if matchup_scores:
           best = np.max(matchup_scores)
           worst = np.min(matchup_scores)
           matchup_score = 0.7 * best + 0.3 * worst
        else:
            matchup_score = 0

        # 🔥 combine (matchup dominates)
        prob = 0.3 * prob + 0.7 * (0.5 + matchup_score)

        results.append((hero, prob))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:10]

In [13]:
hero_map = {
    h['id']: h['localized_name']
    for h in requests.get("https://api.opendota.com/api/heroes").json()
}

In [14]:
def suggest_named(enemy_team):
    results = suggest(enemy_team)

    for h, prob in results:
        print(hero_map.get(h, h), "→", round(prob, 3))

In [15]:
suggest_named([81,82])  

Enchantress → 0.63
Chen → 0.622
Grimstroke → 0.604
Enigma → 0.602
Jakiro → 0.522
Spectre → 0.513
Broodmother → 0.513
Winter Wyvern → 0.512
Night Stalker → 0.511
Razor → 0.51


In [16]:
suggest_named([17, 59, 6, 74])

Elder Titan → 0.718
Primal Beast → 0.673
Lycan → 0.638
Wraith King → 0.545
Night Stalker → 0.536
Phantom Lancer → 0.533
Zeus → 0.532
Lifestealer → 0.532
Spectre → 0.531
Meepo → 0.53


In [17]:
suggest_named([17, 59, 6, 74, 12])

Primal Beast → 0.658
Meepo → 0.633
Lycan → 0.62
Elder Titan → 0.597
Wraith King → 0.522
Night Stalker → 0.519
Zeus → 0.515
Crystal Maiden → 0.515
Spectre → 0.513
Kez → 0.513


In [18]:
import requests

heroes = requests.get("https://api.opendota.com/api/heroes").json()

hero_df = sorted(
    [(h['id'], h['localized_name']) for h in heroes],
    key=lambda x: x[0]
)

for hid, name in hero_df:
    print(f"{hid:3} → {name}")

  1 → Anti-Mage
  2 → Axe
  3 → Bane
  4 → Bloodseeker
  5 → Crystal Maiden
  6 → Drow Ranger
  7 → Earthshaker
  8 → Juggernaut
  9 → Mirana
 10 → Morphling
 11 → Shadow Fiend
 12 → Phantom Lancer
 13 → Puck
 14 → Pudge
 15 → Razor
 16 → Sand King
 17 → Storm Spirit
 18 → Sven
 19 → Tiny
 20 → Vengeful Spirit
 21 → Windranger
 22 → Zeus
 23 → Kunkka
 25 → Lina
 26 → Lion
 27 → Shadow Shaman
 28 → Slardar
 29 → Tidehunter
 30 → Witch Doctor
 31 → Lich
 32 → Riki
 33 → Enigma
 34 → Tinker
 35 → Sniper
 36 → Necrophos
 37 → Warlock
 38 → Beastmaster
 39 → Queen of Pain
 40 → Venomancer
 41 → Faceless Void
 42 → Wraith King
 43 → Death Prophet
 44 → Phantom Assassin
 45 → Pugna
 46 → Templar Assassin
 47 → Viper
 48 → Luna
 49 → Dragon Knight
 50 → Dazzle
 51 → Clockwerk
 52 → Leshrac
 53 → Nature's Prophet
 54 → Lifestealer
 55 → Dark Seer
 56 → Clinkz
 57 → Omniknight
 58 → Enchantress
 59 → Huskar
 60 → Night Stalker
 61 → Broodmother
 62 → Bounty Hunter
 63 → Weaver
 64 → Jakiro
 65 →